# Hybrid and step systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/intro/06_hybrid.ipynb)

Official API intro to the discrete-in-the-loop stack used by digital MPC and
sampled controllers:

1. **`StepSystem`** — discrete leaf (tick logic)
2. **`Computer`** — schedules the step side (`block % dt`)
3. **`HybridDiagram`** — `Computer @ plant` (or `mpc @ plant`) with ZOH + sampling

**Scripts for depth:** `examples/scripts/step/`, `examples/scripts/hybrid/`, `examples/scripts/mpc/`

**Full spatial MPC lab:** [`applications/mpc.ipynb`](../applications/mpc.ipynb)


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


## Minimal MPC closed loop

Continuous plants stay the core. Discrete control uses a small parallel stack:

1. **`StepSystem`** — discrete leaf (tick logic); wire several into a **`StepDiagramSystem`** if needed
2. **`Computer`** — schedules that step side (`block % dt`)
3. **`HybridDiagram`** — `Computer @ plant` (or `mpc @ plant`) with ZOH + sampling; the continuous plant is solved between ticks

The snippet below is the minimal closed-loop pattern (`ModelPredictiveController`
then `mpc @ plant`). Spatial circuit and other workflows live under
`examples/scripts/mpc/` and [applications/mpc.ipynb](../applications/mpc.ipynb).


In [ ]:
import numpy as np

from minilink.control.mpc import ModelPredictiveController
from minilink.core.backends import configure_jax
from minilink.core.costs import QuadraticCost
from minilink.dynamics.catalog.vehicles.jax_vehicles import BicycleDynRate
from minilink.planning.problems import PlanningProblem
from minilink.planning.trajectory_optimization.planner import (
    TrajectoryOptimizationPlanner,
)

configure_jax(enable_x64=True)

U_TARGET = 4.0
TF_SIM = 3.0
MPC_DT = 0.05
SIM_DT = 0.02

plant = BicycleDynRate()
r_r = plant.params["r_r"]
x_ref = np.array([0.0, 0.0, 0.0, U_TARGET, 0.0, 0.0, U_TARGET / r_r, 0.0])
x0 = np.array([0.0, 3.0, 0.0, U_TARGET * 0.8, 0.0, 0.0, (U_TARGET * 0.8) / r_r, 0.0])
plant.x0 = x0.copy()

mpc_planner = TrajectoryOptimizationPlanner(
    PlanningProblem(
        sys=plant,
        tf=2.0,
        x_start=x0,
        cost=QuadraticCost.from_system(
            plant,
            Q=np.diag([0.0, 12.0, 18.0, 0.5, 4.0, 6.0, 0.1, 100.0]),
            R=np.diag([1.0, 25.0]),
            S=np.diag([0.0, 30.0, 40.0, 2.0, 12.0, 18.0, 0.1, 100.0]),
            xbar=x_ref,
            ubar=np.zeros(2),
        ),
    ),
    n_steps=5,
    transcription="direct_collocation",
    compile_backend="jax",
    optimizer_method="scipy_slsqp",
    optimizer_options={"maxiter": 10, "ftol": 1.0},
)

mpc = ModelPredictiveController(
    mpc_planner, dt_mpc=MPC_DT, warm_start=True, step_disp=False
)
hybrid = mpc @ plant
hybrid.plot_diagram()

hybrid.compute_trajectory(
    tf=TF_SIM,
    x0_plant=x0,
    plant_dt_inner=SIM_DT,
    compile_backend="jax",
)
hybrid.plot_trajectory()
